In [1]:
from IPython.display import HTML
HTML("<style>.container{width:100%!important;margin:auto}div.cell,div.input_area{padding:2px;margin:0}div.output_wrapper{padding:0;margin:0}</style>")

In [2]:
import cv2, os, glob, random, time, gc, h5py
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
from scipy import ndimage
from scipy.ndimage import uniform_filter
from scipy.ndimage import sobel
from joblib import Parallel, delayed

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from skimage.morphology import disk
from skimage.filters.rank import median
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from ptflops import get_model_complexity_info

import utils as ut
import models as md
import dataprocess as dp
from uqtools.getClass import *
from uqtools.plotClass import *
from uqtools.getReport import *

device = 'cuda' if torch.cuda.is_available() else 'cpu'
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory / 1024**3, "GB")

# All the models follows No augmentation and attention, common cedice loss and fixed dropout 0.2 (nned to check encoder and decored or decoder only)
# Model 1: Pretrained-Resnet with dropout
# Model 2: Pretrained-mobilenet with dropout
# Model 3: Pretrained-efficientnet7 with dropout
# Model 4: Pretrained-efficientnet7 on UNET++ with dropout
# Model 5: Scratch model without attention(not unet used in pretrained due to dataset constraints)
# Model 6: Scratch model with attention(not unet used in pretrained due to dataset constraints)

NVIDIA H200 NVL 139.80084228515625 GB


In [3]:
#projectnb/labci/Indrajit/Tutorial/Ekha/new_save_model/PretrainRLRPyesM_4C_cediceL_noaug_Dropyes_Final.pth
batch_size = 128
n_models =4
channels = [4,4,4,4,4,4]
n_classes = 2
loss_type = ['cedice','cedice','cedice','cedice','cedice','cedice']
Pretrain = ['yes','yes', 'yes', 'yes', 'no', 'no']
Augmentation = ['noaug', 'noaug', 'noaug','noaug','noaug', 'noaug'] #aug
Dropout = ['yes','yes','yes','yes','yes','yes']
Attention = ['no','no','no','no','no','yes']
pretype = ['PretrainRLRP', 'UMobile_RLRP', 'UEffi_RLRP', 'UPPEffi_RLRP', 'NA']
prename = ['unet_resnet34','unet_mobilenetv2','unet_effb7','unetpp_effb7']
models = []
for i in range(n_models):
    if Pretrain[i] == 'yes':
        model_name = prename[i]   # change here or pass via CLI Null
        model = md.build_model(model_name=model_name, in_channels=3, n_classes=n_classes).to(device)
        if Dropout[i] == 'yes':
            print('Dropout is gonaa be applied.........')
            md.replace_activation_with_dropout(model.decoder, p=0.2)
        save_model_path = f'./new_save_model/{pretype[i]}{Pretrain[i]}M_{channels[i]}C_{loss_type[i]}L_{Augmentation[i]}_Drop{Dropout[i]}_Best.pth'
        print(save_model_path)
        model.load_state_dict(torch.load(save_model_path))
    else:
        if Attention[i] == 'yes': attention_layers=[0,1] # On decoder only
        else: attention_layers=None
        model = md.GeneralizedResidualUNet(in_channels=channels[i], out_channels=n_classes, filters=[32, 64, 128, 256], use_attention='custom',
                                       attention_layers=attention_layers).to(device)
        save_model_path = f'./save_model/MRLRP_{channels[i]}C_{loss_type[i]}L_{Augmentation[i]}_Attn{Attention[i]}_Best.pth'
        print(save_model_path)
        model.load_state_dict(torch.load(save_model_path))
    models.append(model)

Dropout is gonaa be applied.........
./new_save_model/PretrainRLRPyesM_4C_cediceL_noaug_Dropyes_Best.pth
Dropout is gonaa be applied.........
./new_save_model/UMobile_RLRPyesM_4C_cediceL_noaug_Dropyes_Best.pth
Dropout is gonaa be applied.........
./new_save_model/UEffi_RLRPyesM_4C_cediceL_noaug_Dropyes_Best.pth
Dropout is gonaa be applied.........
./new_save_model/UPPEffi_RLRPyesM_4C_cediceL_noaug_Dropyes_Best.pth


In [4]:
path1 = './ETCI_dataset/train/train/'
path2 = './ETCI_dataset/New1/data/test/'
train_samples_all = dp.load_dataset(path1)
#test_samples = dp.load_dataset(path2)
train_samples_all, val_samples = dp.split_by_event(train_samples_all, split_ratio=1.0 - 0.1) #val_split = 0.01
stats = dp.compute_train_stats(train_samples_all) # Compute train stats (do it before augmentation but after train val split)

val_dataset = dp.FloodSARSegDataset(val_samples, stats)
#----------Randomly select 10% samples for UQ---------------#
N = len(val_dataset)
rng = np.random.default_rng(42)  # 42 can be any number
idx = rng.choice(N, size=int(1 * N), replace=False)
val_dataset_subset = Subset(val_dataset, idx)

#test_dataset  = dp.FloodSARSegDataset(test_samples, stats)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader_subset = DataLoader(val_dataset_subset, batch_size=batch_size, shuffle=True, num_workers=4)
#test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)

Found directories: ['bangladesh_20170314t115609', 'bangladesh_20170606t115613', 'bangladesh_20170712t115615', 'nebraska_20170108t002112', 'nebraska_20170213t002121', 'nebraska_20170309t002110', 'nebraska_20170402t002111', 'nebraska_20170508t002113', 'nebraska_20170520t002113', 'nebraska_20170601t002114', 'nebraska_20170731t002118', 'nebraska_20170929t002120', 'nebraska_20171116t002120', 'nebraska_20171210t002119', 'nebraska_20171222t002118', 'northal_20190302t234651', 'northal_20190407t234651', 'northal_20190419t234652', 'northal_20190513t234653', 'northal_20190525t234653', 'northal_20190606t234654', 'northal_20190618t234654', 'northal_20190630t234655', 'northal_20190712t234656', 'northal_20190724t234657', 'northal_20190805t234658', 'northal_20190829t234659', 'northal_20190910t234659', 'northal_20191004t234700', 'northal_20191121t234700', 'northal_20191227t234659']
Total samples loaded from ./ETCI_dataset/train/train/: 33405
Train events: 27, Validation events: 4
Train tiles: 28385, Va

In [6]:
#dropout: for m in model.decoder.modules(): if isinstance(m, (torch.nn.Dropout, torch.nn.Dropout2d)): m.train()
def enable_mc_dropout(model, decoder_only=True, enable_encoder=False):
    model.eval()
    dropout_applied = False
    dropout_applied_encoder = False
    for name, module in model.named_modules():
        if isinstance(module, (nn.Dropout, nn.Dropout2d, nn.AlphaDropout)):
            # Decoder modules
            if decoder_only and ("decoder" in name or "decoders" in name or "up_convs" in name or "attentions" in name):
                module.train()
                dropout_applied = True
            
            # Encoder modules (optional)
            if enable_encoder and ("encoder" in name or "encoders" in name or "bottleneck" in name):
                module.train()
                dropout_applied_encoder = True
    #if dropout_applied: print("Dropout layers in decoder are now enabled for MC Dropout.")
    #else: print(" No dropout layers found in decoder to enable.")
    #if dropout_applied_encoder: print("Dropout layers in encoder are now enabled for MC Dropout.")


def mc_test_evaluation(test_loader, model, channels=4, Pretrain='no', device=None, T=10):
    all_labels, all_preds, all_probs = [], [], []
    model.eval()
    #enable_decoder_dropout(model)
    enable_mc_dropout(model)
    with torch.no_grad():
        for batch in test_loader:
            sar, labels = batch['sar'].to(device), batch['label'].to(device)
            if channels == 2: sar = sar[:, 0:2]
            if Pretrain == 'yes': sar = sar[:, [0, 1, 2], :, :]

            batch_probs = []
            for _ in range(T):
                outputs = model(sar) # Logits
                probs = torch.softmax(outputs, dim=1)
                batch_probs.append(probs)
            
            # Average the probabilities across T passes: (B, C, H, W)
            mean_probs = torch.stack(batch_probs)#.mean(0)
            
            # 4. Final Class Prediction: (B, H, W)
            # Logic: Consensus-based argmax is more stable than single-pass argmax
            preds = torch.argmax(mean_probs.mean(0), dim=1)
            mean_probs = mean_probs.permute(1, 0, 2, 3, 4)  # (N, 5, X, Y)
            #print(labels.shape, preds.shape, mean_probs.shape)#[128, 256, 256] [128, 256, 256]) [128, 5, 2, 256, 256]

            # Keep your existing collection logic
            for i in range(labels.size(0)):
                all_labels.append(labels[i].cpu().byte().numpy())
                all_preds.append(preds[i].cpu().byte().numpy())
                all_probs.append(mean_probs[i].cpu().half().numpy())
            #print(np.array(all_preds).shape, np.array(all_probs).shape)
    return np.stack(all_labels), np.stack(all_preds), np.stack(all_probs)

def compute_entropy_point(probs, eps = 1e-8):
    # Clip for numerical stability
    #probs = probs.astype(np.float64)
    probs = probs / (probs.sum(axis=1, keepdims=True) + eps)
    probs_clipped = np.clip(probs, eps, 1.0-eps)
    entropy = -np.sum(probs_clipped * np.log(probs_clipped), axis=1)  # sum over class dim
    # shape: (N, H, W)
    return entropy

def compute_entropy_ensemble(probs_ens, eps=1e-8):
    # probs_ens: (N, M, 2, H, W)
    #probs_ens = probs_ens.astype(np.float64)
    mean_probs = probs_ens.mean(axis=1)  # (N, 2, H, W)
    pixel_sum = mean_probs.sum(axis=1, keepdims=True)
    mean_probs = np.divide(mean_probs, pixel_sum, out=np.zeros_like(mean_probs), where=pixel_sum != 0)
    mean_probs_clipped = np.clip(mean_probs, eps, 1.0)
    with np.errstate(divide='ignore', invalid='ignore'):
        entropy = -np.sum(mean_probs * np.log(mean_probs), axis=1)
    return np.nan_to_num(entropy)

def flatten_probs_labels(labels, probs):
    y_true = labels.flatten()
    y_prob = probs[:,1,:,:].flatten()  # positive class probability
    return y_true, y_prob
def flatten_ensemble_probs_labels(labels, probs_ens):
    # probs_ens: (N, M, 2, H, W)
    mean_probs = probs_ens.mean(axis=1)  # average over MC runs -> (N, 2, H, W)
    return flatten_probs_labels(labels, mean_probs)

def plot_calibration(y_true, y_prob, n_bins=10, label='Model'):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    plt.plot(prob_pred, prob_true, marker='o', label=label)



bin_edges = np.geomspace(0.01, 0.6, 10)
discard_fractions = np.array([0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5])
model_registry = {}
for idx, m in enumerate(models):
    if idx>=0:
        random.seed(42+idx)
        model_name = f"m{idx}_Pre_{Pretrain[idx]}" # Creates 'm1', 'm2', etc.
        Test_label, Test_eval, Test_prob = mc_test_evaluation(val_loader_subset, m, channels=channels[idx],
                                                              Pretrain = Pretrain[idx], device=device, T=50)
        #--------------- arrange the data for uqtools--------------------#
        Test_label = Test_label.reshape(-1).astype(np.int16)
        Test_prob = np.transpose(Test_prob[:, :, 1, :, :], (0, 2, 3, 1)).astype(np.float32)
        Test_prob = Test_prob.reshape(-1, Test_prob.shape[-1])
        result_dict1 = get_spread_vs_skill(Test_prob, Test_label, bin_edges)
        result_dict2 = get_discard_test(Test_prob, Test_label, discard_fractions)
        result_dict3 = get_reliability_curve_points(observed_labels=Test_label,forecast_probabilities=np.mean(Test_prob, axis=1),
                                                   num_bins=20,climatology=np.mean(Test_label))
        model_registry[model_name] = {'Spread_Skill': result_dict1, 'discard': result_dict2, 'Reliability': result_dict3}        
        #print('---------------------------------------------------------------------------------------------------------')

In [7]:
# Let's plot
model_keys = ['m0_Pre_yes', 'm1_Pre_yes', 'm2_Pre_yes', 'm3_Pre_yes']#,'m4_Pre_no',  'm5_Pre_no']
spread_skills, discards, reliabilities = [], [], []
for key in model_keys:
    spread_skills.append(model_registry[key]['Spread_Skill'])
    discards.append(model_registry[key]['discard'])
    reliabilities.append(model_registry[key]['Reliability'])

model_names = ["URes", "UMobile", 'UEffi', 'UPEffi']#, 'Scratch_NoAttn', 'Scratch_Attn']
fig, _ = plot_spread_vs_skill_curve([spread_skills[0], spread_skills[1], spread_skills[2],spread_skills[3]],
                           figsize=(8,6), model_names=model_names)
fig.savefig("./Figs/spread_vs_skill_curve_main.pdf", dpi=300, bbox_inches='tight') # Common figure for all the models
plt.close(fig)
fig, _ = plot_reliability_curve([reliabilities[0], reliabilities[1], reliabilities[2],reliabilities[3]],
                        figsize=(8,6), model_names=model_names)
fig.savefig("./Figs/reliability_curve_main.pdf", dpi=300, bbox_inches='tight') # Common figure for all the models
plt.close(fig)

In [8]:
def plot_discard_test(result_dict, figsize = (7, 5), line_color='#1f77b4', show_example_fraction=True, MName = 'URes'):
    """
    Plots the discard test: error vs discard fraction. Refined visuals with combined legend and no diagnostic text.
    Parameters
    ----------
    result_dict : dict
        Output from `get_discard_test`
    line_color : str
        Color of the error curve
    show_example_fraction : bool
        If True, overlay fraction of examples remaining on secondary y-axis

    Returns
    -------
    fig, ax : matplotlib Figure and Axes
    """
    discard_frac = result_dict['discard_fractions']
    error_vals = result_dict['error_values']

    fig, ax1 = plt.subplots(figsize=figsize)
    
    # 1. Primary Plot: Error Curve (Left Y-axis)
    line1, = ax1.plot(discard_frac, error_vals, 'o-', color=line_color, 
                      linewidth=2, markersize=6, label='Error', zorder=5)
    
    ax1.set_xlabel('Discard fraction (most uncertain examples removed)', fontsize=16)
    ax1.set_ylabel('Error', fontsize=16)
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.set_title(MName, fontsize=16, fontweight='bold')

    # 2. Secondary Axis: Examples Remaining (Right Y-axis)
    if show_example_fraction:
        example_frac = result_dict['example_fractions']
        ax2 = ax1.twinx()
        # Red dashed line for coverage
        line2, = ax2.plot(discard_frac, example_frac, 's--', color='#d62728', 
                          alpha=0.6, label='Examples left')
        ax2.set_ylabel('Fraction of examples left', color='#d62728', fontsize=16)
        ax2.set_ylim(0, 1.05)
        ax2.tick_params(axis='y', labelcolor='#d62728')
        
        # Merge legends into one box
        lines = [line1, line2]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper right', frameon=True)
    else:
        ax1.legend(loc='upper right', frameon=True)

    plt.tight_layout()
    return fig, ax1

def plot_reliability_curve(result_dicts, figsize = (8, 8), model_names=None, colors=None):
    """
    Compiles multiple models on a single reliability plot without numerical scores.
    """
    # Ensure result_dicts is a list even if a single dict is passed
    if isinstance(result_dicts, dict):
        result_dicts = [result_dicts]
        
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Calibration')

    if colors is None:
        colors = plt.cm.tab10.colors

    for i, rd in enumerate(result_dicts):
        name = model_names[i] if model_names else f"Model {i+1}"
        valid = ~np.isnan(rd['mean_forecast_probs'])
        
        # Removed BSS from the label string
        ax.plot(rd['mean_forecast_probs'][valid], 
                rd['mean_event_frequencies'][valid], 
                'o-', color=colors[i % len(colors)], label=name)

    ax.set_xlabel('Predicted Probability', fontsize=16)
    ax.set_ylabel('Observed Frequency', fontsize=16)
    ax.set_title('(b) Reliability Curve (Comparision)', fontsize=16, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    return fig, ax

def plot_spread_vs_skill_curve(result_dicts, model_names=None, figsize= (7, 7), reference_line=True, colors=None, show_title=True, show_samples=False):
    """
    Plots Spread vs Skill curve: predictive std (spread) vs RMSE (skill).

    Parameters
    ----------
    result_dict : dict
        Output from compute_spread_vs_skill
    reference_line : bool
        If True, plots dashed diagonal: spread = skill
    line_color : str
        Color of main line

    Returns
    -------
    fig, ax : matplotlib Figure and Axes
    """
    # Ensure inputs are lists even if a single model is passed
    if isinstance(result_dicts, dict):
        result_dicts = [result_dicts]
    if model_names is None:
        model_names = [f"Model {i+1}" for i in range(len(result_dicts))]
    elif isinstance(model_names, str):
        model_names = [model_names]
        
    fig, ax = plt.subplots(figsize=figsize)
    
    # 1. Determine Global Plot Limits (to keep 1:1 ratio consistent for all models)
    global_max = 0
    for rd in result_dicts:
        m_max = np.nanmax(rd['mean_prediction_stds'])
        r_max = np.nanmax(rd['rmse_values'])
        global_max = max(global_max, m_max, r_max)
    
    limit_val = global_max * 1.1 if global_max > 0 else 0.5
    
    # 2. Draw Reference Background
    if reference_line:
        ax.plot([0, limit_val], [0, limit_val], '--', color='black', alpha=0.6, label='Ideal (Spread=Skill)')
        ax.fill_between([0, limit_val], [0, limit_val], limit_val, color='red', alpha=0.03, label='Overconfident')
        ax.fill_between([0, limit_val], 0, [0, limit_val], color='blue', alpha=0.03, label='Underconfident')

    # 3. Setup Colors
    if colors is None:
        # Use a standard colormap for multiple models
        cmap = plt.cm.get_cmap('tab10')
        colors = [cmap(i) for i in range(len(result_dicts))]
    elif isinstance(colors, str):
        colors = [colors]

    # 4. Plot each model
    for rd, name, color in zip(result_dicts, model_names, colors):
        mean_stds = rd['mean_prediction_stds']
        rmse = rd['rmse_values']
        
        valid = ~np.isnan(mean_stds) & ~np.isnan(rmse)
        
        ax.plot(mean_stds[valid], rmse[valid], 'o-', color=color, markersize=7, 
                linewidth=2, label=name, zorder=5)
    
    # 5. Optional Statistical Annotation (Summary of all N)
    if show_samples:
        total_n = sum([np.sum(rd['example_counts']) for rd in result_dicts])
        ax.text(0.05, 0.95, f'Total samples (all): {total_n:,}', transform=ax.transAxes, 
                fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # Formatting
    ax.set_xlabel('Spread (Predictive Standard Deviation)', fontsize=16)
    ax.set_ylabel('Skill (Actual RMSE)', fontsize=16)
    
    if show_title:
        title = '(a) Spread-Skill Curve (Comparision)' if len(result_dicts) > 1 else 'Spread-Skill Reliability'
        ax.set_title(title, fontsize=16, fontweight='bold')
    
    ax.set_xlim(0, limit_val-0.1)
    ax.set_ylim(0, limit_val)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # Move legend smartly
    n_models = len(result_dicts)
    if n_models <= 7: # change the number to get legend at the bottom
        ax.legend(loc='lower right', frameon=True)
    else:
        ncols = min(n_models, 4)
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15),
                  ncol=ncols, frameon=True)
    
    plt.tight_layout()
    return fig, ax

In [9]:
SS, D, R = [], [], []
sequence = ['c','d','e','f']
for i, key in enumerate(model_names):
    #-----------Get the health report and store it-----------------------#
    SS.append(spread_vs_skill_health(spread_skills[i])), D.append(discard_health(discards[i]))
    R.append(reliability_health(reliabilities[i]))
    #-----------Plot function-------------------------------------------#
    
    fig, _ = plot_example_histogram(spread_skills[i], figsize=(6,6), color='skyblue', show_title=True, show_bar_labels=False)
    fig.savefig(f"./Figs/{model_names[i]}/histogram_main.pdf", dpi=300, bbox_inches='tight') # Common figure for all the models
    plt.close(fig)
    fig, _ = plot_mean_pred_vs_target(spread_skills[i], figsize=(8,6), color_pred='blue', color_target='red', show_title=True)
    fig.savefig(f"./Figs/{model_names[i]}/mean_pred_vs_target_main.pdf", dpi=300, bbox_inches='tight') # Common figure for all the models
    plt.close(fig)
    fig, _ = plot_spread_vs_rmse_bar(spread_skills[i], figsize=(8,6), show_title=True, show_bar_labels=True)
    fig.savefig(f"./Figs/{model_names[i]}/spread_vs_rmse_bar_main.pdf", dpi=300, bbox_inches='tight') # Common figure for all the models
    plt.close(fig)
    fig, _ = plot_discard_test(discards[i], figsize=(8,6), line_color='#1f77b4', MName = f'({sequence[i]}) Discard Curve ({model_names[i]})', show_example_fraction=True)
    fig.savefig(f"./Figs/{model_names[i]}/discard_main.pdf", dpi=300, bbox_inches='tight') # Common figure for all the models
    plt.close(fig)
    fig, _ = plot_attributes_diagram(reliabilities[i], figsize=(8,6), model_name=model_names[i], figure=None, axes=None)
    fig.savefig(f"./Figs/{model_names[i]}/attributes_main.pdf", dpi=300, bbox_inches='tight') # Common figure for all the models
    plt.close(fig)

In [10]:
# Print the UQ health report
def print_model_comparison_table(model_names, SS_results, D_results, R_results):
    """
    Prints a formatted table comparing models across all diagnostic dimensions.
    """

    header = (
        f"{'Model':<15} | "
        f"{'SS-Rat':<7} | {'SS-Rel':<7} | {'SS-Corr':<7} | "
        f"{'Mono-Frac':<10} | {'DI':<6} | {'DI-Avg':<7} | "
        f"{'Init-Err':<9} | {'Final-Err':<9} | "
        f"{'CRPS/BS':<8} | {'BSS':<6} | {'ECE':<6}"
    )
    separator = "-" * len(header)

    print("\n" + "=" * 60)
    print("           MULTI-MODEL DIAGNOSTIC SUMMARY")
    print("=" * 60)
    print(header)
    print(separator)

    for i, model in enumerate(model_names):

        # -------- Spread–Skill --------
        ss = SS_results[i]
        ss_rat  = f"{ss.get('ssrat', float('nan')):.3f}"
        ss_rel  = f"{ss.get('ssrel', float('nan')):.3f}"
        ss_corr = (
            f"{ss['correlation']:.3f}"
            if ss.get('correlation') is not None else "N/A"
        )

        # -------- Discard --------
        d = D_results[i]
        mf        = f"{d.get('mf', 0):.2%}"
        di        = f"{d.get('di', float('nan')):.3f}"
        di_avg   = f"{d.get('di_avg', float('nan')):.3f}"
        init_err = f"{d.get('initial_error', float('nan')):.3f}"
        fin_err  = f"{d.get('final_error', float('nan')):.3f}"

        # -------- Reliability --------
        r = R_results[i]
        bs  = f"{r.get('crps_bs', float('nan')):.4f}"
        bss = f"{r.get('bss', float('nan')):.3f}"
        ece = f"{r.get('ece', float('nan')):.3f}"

        # -------- Row --------
        print(
            f"{model:<15} | "
            f"{ss_rat:<7} | {ss_rel:<7} | {ss_corr:<7} | "
            f"{mf:<10} | {di:<6} | {di_avg:<7} | "
            f"{init_err:<9} | {fin_err:<9} | "
            f"{bs:<8} | {bss:<6} | {ece:<6}"
        )

    print(separator + "\n")


# --- Example Call ---
model_names = list(model_registry.keys())
print(model_names)
print_model_comparison_table(model_names, SS, D, R)

['m0_Pre_yes', 'm1_Pre_yes', 'm2_Pre_yes', 'm3_Pre_yes']

           MULTI-MODEL DIAGNOSTIC SUMMARY
Model           | SS-Rat  | SS-Rel  | SS-Corr | Mono-Frac  | DI     | DI-Avg  | Init-Err  | Final-Err | CRPS/BS  | BSS    | ECE   
----------------------------------------------------------------------------------------------------------------------------------
m0_Pre_yes      | 0.103   | 0.056   | 0.867   | 100.00%    | 66.378 | 0.003   | 0.048     | 0.016     | 0.0068   | 0.625  | 0.003 
m1_Pre_yes      | 0.131   | 0.069   | 0.951   | 100.00%    | 74.187 | 0.005   | 0.066     | 0.017     | 0.0085   | 0.537  | 0.002 
m2_Pre_yes      | 0.069   | 0.059   | 0.862   | 40.00%     | 50.223 | 0.004   | 0.074     | 0.037     | 0.0080   | 0.560  | 0.006 
m3_Pre_yes      | 0.127   | 0.066   | 0.902   | 100.00%    | 74.360 | 0.005   | 0.064     | 0.016     | 0.0093   | 0.489  | 0.006 
------------------------------------------------------------------------------------------------------------------

In [11]:
print(f"{'Model':<8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'IoU':>8} {'Acc':>8}")
print("-" * 56)
for idx, m in enumerate(models):
    if idx>=0:
        model_name = f"m{idx}_Pre_{Pretrain[idx]}" # Creates 'm1', 'm2', etc.
        Test_label, Test_eval, Test_prob = mc_test_evaluation(val_loader_subset, m, channels=channels[idx], Pretrain = Pretrain[idx], device=device, T=20)
        y_true_t = torch.tensor(Test_label, device=device)
        y_pred_t = torch.tensor(Test_eval, device=device)
        metric = ut.evaluate_segmentation(y_pred_t, y_true_t, n_classes=2)
        print(f"{model_name} "f"{metric['mean_precision']:>8.3f} "f"{metric['mean_recall']:>8.3f} "f"{metric['mean_f1']:>8.3f} "
              f"{metric['mean_iou']:>8.3f} "f"{metric['overall_accuracy']:>8.3f}")

Model        Prec   Recall       F1      IoU      Acc
--------------------------------------------------------
m0_Pre_yes    0.950    0.816    0.871    0.794    0.992
m1_Pre_yes    0.933    0.792    0.848    0.765    0.991
m2_Pre_yes    0.886    0.843    0.863    0.783    0.991
m3_Pre_yes    0.893    0.800    0.840    0.755    0.990


In [12]:
from ptflops import get_model_complexity_info
for model in models:
    model = model.cpu()  # Move to CPU
    model.eval()
    macs, params = get_model_complexity_info(
        model, (1, 3, 256, 256),  # Your 4-ch SAR
        as_strings=True, 
        print_per_layer_stat=False,
        input_constructor=lambda _: torch.randn(1, 3, 256, 256)
    )
    print(f"4-ch SAR U-Net: {macs} | {params}")
    print('----------------------------------------------------------------------------------------------------')

4-ch SAR U-Net: 7.86 GMac | 24.44 M
----------------------------------------------------------------------------------------------------
4-ch SAR U-Net: 3.4 GMac | 6.63 M
----------------------------------------------------------------------------------------------------
4-ch SAR U-Net: 3.15 GMac | 67.1 M
----------------------------------------------------------------------------------------------------
4-ch SAR U-Net: 12.19 GMac | 68.16 M
----------------------------------------------------------------------------------------------------
